In [ ]:
import os
import re
from ase import Atoms
from ase.units import Ry
from ase.visualize import view
from ase.calculators.siesta import Siesta
from ase.calculators.siesta.parameters import Species, PAOBasisBlock
from src.structure import Perovskite, get_reduced_formula
from src.cleanfiles import cleanFiles
from src.fdfcreate import generate_basis

In [ ]:
perov = Perovskite('BaTiO3', bulk=False, N=1.5)
atoms_test = perov.atoms

In [ ]:
test_atom = Atoms(symbols='Ba')
test_atom.center(vacuum=5.0)
#test_atom
view(test_atom)

In [ ]:
#generate_basis(test_atom, basis='DZPp', dir='resultsold/ML')

In [ ]:
def run_test_calc(atoms, xcf='PBEsol', basis='DZP', EnergyShift=0.01, SplitNorm=0.15,
                  MeshCutoff=200, kgrid=(10, 10, 10), dir=''):

    # Define current working directory and extract information from the ASE Atoms object
    cwd = os.getcwd()
    formula = get_reduced_formula(atoms)

    if basis.endswith('p'):
        basis = basis[:-1]

    # Calculation parameters in a dictionary
    calc_params = {
        'label': f'{formula}',
        'xc': xcf,
        'basis_set': basis,
        'mesh_cutoff': MeshCutoff * Ry,
        'energy_shift': EnergyShift * Ry,
        'kpts': kgrid,
        'directory': dir,
        'pseudo_path': os.path.join(cwd, 'pseudos', f'{xcf}')
    }
    dir_fdf = os.path.join(cwd, dir)
    # fdf arguments in a dictionary
    fdf_args = {
        '%include': os.path.join(dir_fdf, 'basis.fdf'),
        'PAO.SplitNorm': SplitNorm,
        #'kgrid.Cutoff': '0. Bohr'
        #'SCF.DM.Tolerance': 1e-6
    }
    # Set up the Siesta calculator
    calc = Siesta(**calc_params, fdf_arguments=fdf_args)
    atoms.calc = calc
    # Perform a single-point calculation to generate the .fdf file and extract the basis block
    energy = atoms.get_potential_energy()

    # Remove intermediate files generated by the calculation, keeping only the important ones
    cleanFiles(directory=dir, confirm=False)

    return energy

In [ ]:
#test_atom.center(vacuum=15)
run_test_calc(test_atom, xcf='PBEsol', basis='DZP', MeshCutoff=100, dir='resultsold/ML')

In [ ]:
def calculate_atomic_energies(formula, xcf='PBEsol', basis='DZP',
                              EnergyShift=0.01, SplitNorm=0.15,
                              MeshCutoff=200, dir=''):

    symbols = re.findall(r'[A-Z][a-z]*', formula)

    energies = {}
    for j, symbol in enumerate(symbols):
        dir_symbol = os.path.join(dir, symbol)
        atom = Atoms(symbols=symbol)
        atom.center(vacuum=5.0)
        # If basis ends with 'p' and this is not the first atom, remove the 'p' to avoid generating a polarized basis for subsequent atoms
        if j != 0 and basis.endswith('p'):
            basis = basis[:-1]
        # Generate the basis set for the test atom and save it in the specified directory
        generate_basis(atom, xcf, basis, EnergyShift, SplitNorm, dir=dir_symbol)
        # Add vacuum to the atom to ensure it is isolated for the calculation
        atom.center(vacuum=15.0)
        # Run the test calculation for the atom and store the energy in the dictionary
        energy = run_test_calc(atom, xcf, basis, EnergyShift, SplitNorm,
                               MeshCutoff, kgrid=(1, 1, 1), dir=dir_symbol)
        energies[symbol] = energy

    return energies


In [ ]:
en = calculate_atomic_energies('BaTiO3', xcf='PBEsol', bbasis='DZP',
                               MeshCutoff=200, dir='resultsold/ML')

In [ ]:
en